# Representational Alignment Metrics

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
import gc
import shutil
from tqdm.notebook import tqdm

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import pdist, squareform

import seaborn as sns
import matplotlib.pyplot as plt

import itertools

In [ ]:
folders = [
    'frame_base_layers', 'frame_large_layers', 'gen_base_layers', 'gen_large_layers',
    'iter3_layers', 'iter3_plus_layers', 'qwen_layers', 'results'
]

# make folders
for folder in folders:
  os.makedirs(folder, exist_ok=True)

# upload embeddings to colab

In [ ]:
# set text size for figures when adding to paper
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

In [ ]:
# reset figure text sizes to default

# import matplotlib as mpl
# mpl.rcdefaults()

## Prepare Functions

In [ ]:
def mat_to_df(matrix, index, columns):
  df = pd.DataFrame(
      matrix,
      index=[f"{index}_{i}" for i in range(matrix.shape[0])],
      columns=[f"{columns}_{j}" for j in range(matrix.shape[1])]
  ).iloc[::-1]

  return df

### Linear Predictivity

In [ ]:
# Functions from Zoe
def get_device(use_gpu=True):
    """
    set the device for the metrics
    """
    return torch.device("cuda" if use_gpu and torch.cuda.is_available() else "cpu")

def to_device(data, device):
    """
    move the data to the chosen device.
    NOTE: this also handles numpy array to avoid errors from directly applying .to(device).
    """
    #print(data)
    if isinstance(data, np.ndarray): # if it is in numpy, make sure to convert to torch tensor first
        return torch.tensor(data, device=device)
    elif isinstance(data, torch.Tensor):
        return data.to(device)
    else:
        raise TypeError("Input must be a numpy array or a torch tensor.")


def LinearPredictivity(X, Y):
    """
    evaluate linear predictivity by fitting a ridge regression (with cross-validation)
    and then computing the average pairwise correlation between the predictions and the true values.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # split into train and test regression
    all_corrs=[]
    kf = KFold(n_splits=5,  shuffle=True, random_state = 42)
    for i, (train_idx, test_idx) in enumerate(kf.split(X)):
        #train regression
        predictor = RidgeCV(alphas=np.logspace(-8,8,17))
        predictor.fit(normalize(X[train_idx]), Y[train_idx])

        # test predictions
        y_pred = predictor.predict(normalize(X[test_idx]))

        corr=many_pairwise_correlation(y_pred,Y[test_idx], device=device)
        del predictor
        gc.collect()
        all_corrs.append(corr)

    return all_corrs



###### Torch-based Metrics ########
def pairwise_correlation(A, B, device=None):
    """
    compute the pairwise correlation between each feature in A and each feature in B.

    @params:
        - A, B (numpy array or torch tensor): 2d matrices with dim (N, feats_A) and (N, feats_D)
        - device: torch.device to use for computation. If None (default), get_device() is used to get cuda if available.

    @return:
        - A torch tensor containing pairwise Pearson correlations.
        """
    if device is None:
        device = get_device()
    A = to_device(A, device).to(torch.float32)
    B = to_device(B, device).to(torch.float32)
    # first compute the mean activation across all samples (dim=0) for each feature dimension (dim=1)
    # subtract the original matrix with the mean to center the data across samples
    am = A - torch.mean(A, dim=0, keepdim=True)
    bm = B - torch.mean(B, dim=0, keepdim=True)
    # outline: for each pair of features: A_i and B_j:
        # compute cov(A_i, B_j) = sum_{k=1}^n (A_ki - am_i) (B_kj - bm_j), where k is the k'th sample
        # that is, to compare how similar the relative (centered) behaviors of A_i and B_j are for n samples
        # (aka whether both have relatively larger activation for sample k and smaller activation for sample k')
        # normalize with the standard deviations of each A_i and B_j
    return am.T @ bm / ( # shape: (num_features_A, num_features_B)
        torch.sqrt(torch.sum(am**2, dim=0, keepdim=True)).T *
        torch.sqrt(torch.sum(bm**2, dim=0, keepdim=True)))


def many_pairwise_correlation(A, B, device=None):
    """
    Compute the average of the diagonal entries (across features) of the pairwise correlation
    matrix in memory-efficient chunks.

    NOTE:
    - this assumes that each feature in A corresponds directly to a feature in B.
    - assumption of one-to-one correspondence (one fature in A is meant to align with the a specific feature
        in B).

    A, B: np.ndarray or torch.Tensor
    device: torch.device (optional). If not provided, get_device() is used.
    """
    if device is None:
        device = get_device()

    memsize = 10000 # number of features to process at a time
    corr = []
    lens = []
    # Process the feature dimensions in chunks
    for i in range(0, A.shape[1], memsize):
        # slice the chunk for A and B before putting on the proper device (to save cpu memory)
        seg_A_t = to_device(A[:, i:i+memsize], device)
        seg_B_t = to_device(B[:, i:i+memsize], device)

        model_corrs = pairwise_correlation(seg_A_t, seg_B_t, device=device)
        # move the result back to cpu and convert to numpy for further processing.
        model_corrs_cpu = model_corrs.cpu().numpy()
        # get the diagonal values (pairwise correlation for only corresponding features i)
        # get a single average for this chunk
        corr.append(np.nanmean(np.diag(model_corrs_cpu)))
        lens.append(seg_A_t.shape[1]) # keep track of how many features processed in this chunk
    # compute the weighted average of per-chunk diagonal means
    corr = np.sum(np.multiply(np.array(corr), np.array(lens))) / np.sum(lens)
    gc.collect()
    return corr



def normalize(X):
    """
    this function normalizes the representation along rows (along all samples)
    """
    mean = np.nanmean(X, 0) # training set

    stddev = np.nanstd(X, 0) # training set
    X_zm = X - mean
    X_zm_unit = X_zm / stddev
    X_zm_unit[np.isnan(X_zm_unit)] = 0
    return X_zm_unit

In [ ]:
# using Zoe's functions

def mean_alignment(vision_layer, audio_layer):
    """
    Compute the linear predictivity score both ways and take the mean
    """
    res_vision = LinearPredictivity(vision_layer, audio_layer)
    res_audio = LinearPredictivity(audio_layer, vision_layer)

    mean_alignment = np.mean([np.mean(res_vision), np.mean(res_audio)])

    return mean_alignment

def lin_pred_matrix(embeddings_1, embeddings_2):
    """
    Compute mean linear predictivity scores for all layer combinations

    embeddings_1, embeddings_2: lists containing embedding layers
    """
    matrix = np.array([
        [mean_alignment(layer_1, layer_2) for layer_1 in embeddings_1] for layer_2 in tqdm(embeddings_2, desc="Outer loop")
      ])
    return matrix

In [ ]:
### new functions ###

def fit_normalization(X):
  mean = np.nanmean(X, axis=0)
  std = np.nanstd(X, axis=0)

  # avoid dividing by zero
  std[std==0] = 1

  return mean, std

def apply_normalization(X, mean, std):
  Xn = (X - mean) / std
  Xn[np.isnan(Xn)] = 0
  return Xn

def bidirectional_predictivity(X, Y, train_idx, test_idx, device=None, alphas=np.logspace(-8, 8, 17)):
  """
  Compute bidirectional linear predictivity (X --> Y, Y --> X)

  Returns:
      mean of both directions
  """

  # split
  X_train = X[train_idx]
  X_test = X[test_idx]

  Y_train = Y[train_idx]
  Y_test = Y[test_idx]

  # normalize using train only
  mx, sx = fit_normalization(X_train)
  X_train = apply_normalization(X_train, mx, sx)
  X_test = apply_normalization(X_test, mx, sx)

  my, sy = fit_normalization(Y_train)
  Y_train = apply_normalization(Y_train, my, sy)
  Y_test = apply_normalization(Y_test, my, sy)

  # X --> Y
  predictor_xy = RidgeCV(alphas=alphas)
  predictor_xy.fit(X_train, Y_train)
  pred_Y = predictor_xy.predict(X_test)
  corr_xy = many_pairwise_correlation(
      pred_Y,
      Y_test,
      device=device
  )

  # Y --> X
  predictor_yx = RidgeCV(alphas=alphas)
  predictor_yx.fit(Y_train, X_train)
  pred_X = predictor_yx.predict(Y_test)
  corr_yx = many_pairwise_correlation(
      pred_X,
      X_test,
      device=device
  )

  del predictor_xy
  del predictor_yx
  gc.collect()

  return (corr_xy + corr_yx) / 2

def lin_pred_matrix_bidirectional(layers1, layers2, n_splits=5):
  """
  Compute bidirectional linear predictivity matrix.

  Returns:
    matrix of shape (len(layers1), len(layers2))
  """
  device = get_device()

  num1 = len(layers1)
  num2 = len(layers2)

  num_stimuli = layers1[0].shape[0]

  kf = KFold(
      n_splits=n_splits,
      shuffle=True,
      random_state=42
  )

  fold_results = np.zeros((n_splits, num1, num2), dtype=np.float32)

  for fold_idx, (train_idx, test_idx) in enumerate(kf.split(np.arange(num_stimuli))):
    print(f"Processing Fold {fold_idx + 1}")

    for i in tqdm(range(num1), desc='Outer Loop', leave=False):
      X = layers1[i]

      for j in tqdm(range(num2), desc='Inner Loop', leave=False):
        Y = layers2[j]

        score = bidirectional_predictivity(X, Y, train_idx, test_idx, device=device)

        fold_results[fold_idx, i, j] = score

  final_matrix = np.mean(fold_results, axis=0)

  return final_matrix

In [ ]:
def lp_fig(df, title, xtitle, ytitle, annot=True, figsize=(18,7), color_palette='viridis', decimals='.2f', annot_size=10, legend_size=12):
    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        df,
        cmap=color_palette,
        annot=annot,
        fmt=decimals,
        annot_kws={'size': annot_size}
    )
    ax.set_xticklabels(range(df.shape[1]), rotation=0, fontsize=10)
    ax.set_yticklabels(range(df.shape[0])[::-1], rotation=0, fontsize=10)
    plt.title(title, fontsize=20)
    plt.xlabel(xtitle, fontsize=15, labelpad=15)
    plt.ylabel(ytitle, fontsize=15)
    plt.legend(fontsize=legend_size)
    plt.show()

### Representational Similarity Analysis (RSA)

In [ ]:
# RSA Functions
def compute_rdm(embeddings, metric='correlation'):
    """
    Computes a Representational Dissimilarity Matrix (RDM)
    """
    # flatten if layer is multi-dimensional
    if embeddings.ndim > 2:
        embeddings = embeddings.reshape(embeddings.shape[0], -1)

    # compute pairwise distances
    pairwise_dist = pdist(embeddings, metric=metric)

    # convert one-dimensional vector into N x N matrix
    rdm = squareform(pairwise_dist)

    return rdm

def compare_rdms(rdm1, rdm2, metric='spearman'):
    """
    Compare two RDMs by correlating their upper triangles
    """
    # extract upper triangle indices (excluding diagonal)
    idx = np.triu_indices(rdm1.shape[0], k=1)

    vector1 = rdm1[idx]
    vector2 = rdm2[idx]

    if metric == 'spearman':
        score, p_value = spearmanr(vector1, vector2)
    elif metric == 'pearson':
        score, p_value = pearsonr(vector1, vector2)
    else:
        raise ValueError("Metric must be 'spearman' or 'pearson'.")

    return score, p_value


def rsa_matrix(layers1, layers2, distance_metric='correlation', compare_metric='spearman', normalize_fn=None):
    """
    Computes RSA similarity matrix between all layers
    """
    num1 = len(layers1)
    num2 = len(layers2)

    # initialize empty grid for correlation scores
    similarity_grid = np.zeros((num1, num2))

    # normalization
    if normalize_fn is not None:
        layers1 = [normalize_fn(x) for x in layers1]
        layers2 = [normalize_fn(x) for x in layers2]

    # compute rdms
    rdms1 = [compute_rdm(x, metric=distance_metric) for x in layers1]
    rdms2 = [compute_rdm(x, metric=distance_metric) for x in layers2]

    # comparing rdms for all layers
    for a_idx in range(num1):
        for b_idx in range(num2):
            score, _ = compare_rdms(rdms1[a_idx], rdms2[b_idx], metric=compare_metric)
            similarity_grid[a_idx, b_idx] = score

    return similarity_grid

In [ ]:
def rsa_fig(df, title, xtitle, ytitle, figsize=(18,7), color_palette='magma', annot=True, decimals='.2f', annot_size=10, legend_size=12):
    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        df,
        cmap=color_palette,
        annot=annot,
        fmt=decimals,
        cbar_kws={'label': "Spearman Correlation (\u03C1)"},
        annot_kws={'size': annot_size}
    )
    ax.set_xticklabels(range(df.shape[1]), rotation=0, fontsize=10)
    ax.set_yticklabels(range(df.shape[0])[::-1], rotation=0, fontsize=10)
    plt.title(title, fontsize=20)
    plt.xlabel(xtitle, fontsize=15, labelpad=15)
    plt.ylabel(ytitle, fontsize=15)
    plt.legend(fontsize=legend_size)
    plt.show()

# Load Embeddings

In [ ]:
# check that ordering is the same for dfferent stimulus types and models
frame_base_ids = np.load("frame_base_layers/ids.npy")
frame_large_ids = np.load("frame_large_layers/ids.npy")
gen_base_ids = np.load("gen_base_layers/ids.npy")
gen_large_ids = np.load("gen_large_layers/ids.npy")
audio3_ids = np.load("iter3_layers/ids.npy")
audio3_plus_ids = np.load("iter3_plus_layers/ids.npy")
qwen3_ids = np.load("qwen_layers/ids.npy")

embedding_ids = [frame_base_ids, frame_large_ids, gen_base_ids, gen_large_ids, audio3_ids, audio3_plus_ids, qwen3_ids]

for a, b in itertools.combinations(embedding_ids, 2):
    assert np.array_equal(a, b), "Arrays not equal"

FileNotFoundError: [Errno 2] No such file or directory: 'frame_base_layers/ids.npy'

In [ ]:
# number of layers
dinov2_base = len(os.listdir("frame_base_layers")) - 1
dinov2_large = len(os.listdir("frame_large_layers")) - 1
beats3 = len(os.listdir("iter3_layers")) - 1
beats3_plus = len(os.listdir("iter3_plus_layers")) - 1
qwen3_17b = len(os.listdir("qwen3_1.7b_layers")) - 1

In [ ]:
# load layer embeddings
frame_base = [np.load(os.path.join("frame_base_layers", f"layer_{v}.npy")) for v in range(dinov2_base)]
frame_large = [np.load(os.path.join("frame_large_layers", f"layer_{v}.npy")) for v in range(dinov2_large)]
gen_base = [np.load(os.path.join("gen_base_layers", f"layer_{v}.npy")) for v in range(dinov2_base)]
gen_large = [np.load(os.path.join("gen_large_layers", f"layer_{v}.npy")) for v in range(dinov2_large)]
audio_3 = [np.load(os.path.join("iter3_layers", f"layer_{v}.npy")) for v in range(beats3)]
audio_3_plus = [np.load(os.path.join("iter3_plus_layers", f"layer_{v}.npy")) for v in range(beats3_plus)]
qwen3 = [np.load(os.path.join("qwen_layers", f"layer_{v}.npy")) for v in range(qwen3_17b)]

In [ ]:
# type and shape of embeddings
print(f"dinov2 base embedding type: {type(frame_base[0])}\n # of dinov2 base layers: {dinov2_base}\n dinov2 base layer embedding shape: {frame_base[0].shape}\n")
print(f"dinov2 large embedding type: {type(frame_large[0])}\n # of dinov2 large layers: {dinov2_large}\n dinov2 large layer embedding shape: {frame_large[0].shape}\n")
print(f"beats iter3 embedding type: {type(audio_3[0])}\n # of beats iter3 layers: {beats3}\n beats iter3 layer embedding shape: {audio_3[0].shape}\n")
print(f"beats iter3 plus embedding type: {type(audio_3_plus[0])}\n # of beats iter3 plus layers: {beats3_plus}\n beats iter3 plus layer embedding shape: {audio_3_plus[0].shape}\n")
print(f"qwen3-1.7b embedding type: {type(qwen3[0])}\n # of qwen3-1.7b layers: {qwen3_17b}\n qwen3-1.7b layer embedding shape: {qwen3[0].shape}")

# Vision and Language Alignment

## Linear Predictivity

In [ ]:
vision_lang_lp = {
    'base_gen_images': lin_pred_matrix(qwen3, gen_base),
    'base_mid_frames': lin_pred_matrix(qwen3, frame_base),
    'large_gen_images': lin_pred_matrix(qwen3, gen_large),
    'large_mid_frames': lin_pred_matrix(qwen3, frame_large)
}

In [ ]:
print(
    vision_lang_lp['base_gen_images'].shape,
    vision_lang_lp['base_mid_frames'].shape,
    vision_lang_lp['large_gen_images'].shape,
    vision_lang_lp['large_mid_frames'].shape
)

In [ ]:
# dataframes for visualizations
gen_base_lang_lp = mat_to_df(vision_lang_lp['base_gen_images'], index='vision', columns='lang')

frame_base_lang_lp = mat_to_df(vision_lang_lp['base_mid_frames'], index='vision', columns='lang')

gen_large_lang_lp = mat_to_df(vision_lang_lp['large_gen_images'], index='vision', columns='lang')

frame_large_lang_lp = mat_to_df(vision_lang_lp['large_mid_frames'], index='vision', columns='lang')

# save dataframes
gen_base_lang_lp.to_csv('results/gen_base_qwen1.7b_lp.csv', index=True)
frame_base_lang_lp.to_csv('results/frame_base_qwen1.7b_lp.csv', index=True)
gen_large_lang_lp.to_csv('results/gen_large_qwen1.7b_lp.csv', index=True)
frame_large_lang_lp.to_csv('results/frame_large_qwen1.7b_lp.csv', index=True)

In [ ]:
# heatmaps
lp_fig(gen_base_lang_lp, title='Generated Images LP', xtitle='Qwen3-1.7B', ytitle='DINOv2 Base')

lp_fig(frame_base_lang_lp, title='Middle Frames LP', xtitle='Qwen3-1.7B', ytitle='DINOv2 Base')

lp_fig(gen_large_lang_lp, title='Generated Images LP', xtitle='Qwen3-1.7B', ytitle='DINOv2 Large')

lp_fig(frame_large_lang_lp, title='Middle Frames LP', xtitle='Qwen3-1.7B', ytitle='DINOv2 Large')

In [ ]:
# heatmaps for paper

# vision language alignment
# gen images base | frame base
# gen images large | frame large

vision_lang_dfs = [gen_base_lang_lp, frame_base_lang_lp, gen_large_lang_lp, frame_large_lang_lp]

vmin = min(df.min().min() for df in vision_lang_dfs)
vmax = max(df.max().max() for df in vision_lang_dfs)

fig, axes = plt.subplots(2, 2, figsize=(16,8))

# top left
sns.heatmap(gen_base_lang_lp, ax=axes[0,0], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[0,0].set_title('Generated Images')
axes[0,0].set_xlabel('Qwen3-1.7B')
axes[0,0].set_ylabel('DINOv2 Base')

xlabels = [tick.get_text() for tick in axes[0,0].get_xticklabels()]
new_xlabels = [label.replace("lang_", "") for label in xlabels]
axes[0,0].set_xticklabels(new_xlabels, rotation=0)

base_ylabels = [tick.get_text() for tick in axes[0,0].get_yticklabels()]
new_base_ylabels = [label.replace('vision_', '') for label in base_ylabels]
axes[0,0].set_yticklabels(new_base_ylabels, rotation=0)

# top right
sns.heatmap(frame_base_lang_lp, ax=axes[0,1], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[0,1].set_title('Middle Frames')
axes[0,1].set_xlabel('Qwen3-1.7B')
axes[0,1].set_ylabel('DINOv2 Base')
axes[0,1].set_xticklabels(new_xlabels, rotation=0)
axes[0,1].set_yticklabels(new_base_ylabels, rotation=0)

# bottom left
sns.heatmap(gen_large_lang_lp, ax=axes[1,0], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[1,0].set_title('Generated Images')
axes[1,0].set_xlabel('Qwen3-1.7B')
axes[1,0].set_ylabel('DINOv2 Large')
axes[1,0].set_xticklabels(new_xlabels, rotation=0)

large_ylabels = [tick.get_text() for tick in axes[1,0].get_yticklabels()]
new_large_ylabels = [label.replace('vision_', '') for label in large_ylabels]
axes[1,0].set_yticklabels(new_large_ylabels, rotation=0)

# bottom right
sns.heatmap(frame_large_lang_lp, ax=axes[1,1], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[1,1].set_title('Middle Frames')
axes[1,1].set_xlabel('Qwen3-1.7B')
axes[1,1].set_ylabel('DINOv2 Large')
axes[1,1].set_xticklabels(new_xlabels, rotation=0)
axes[1,1].set_yticklabels(new_large_ylabels, rotation=0)

plt.tight_layout()
plt.show()

## RSA

In [ ]:
vision_lang_rsa = {
    'base_gen_images': rsa_matrix(gen_base, qwen3),
    'base_mid_frames': rsa_matrix(frame_base, qwen3),
    'large_gen_images': rsa_matrix(gen_large, qwen3),
    'large_mid_frames': rsa_matrix(frame_large, qwen3)
}

In [ ]:
# dataframes for visualizations
gen_base_lang_rsa = mat_to_df(vision_lang_rsa['base_gen_images'], index='vision', columns='lang')

frame_base_lang_rsa = mat_to_df(vision_lang_rsa['base_mid_frames'], index='vision', columns='lang')

gen_large_lang_rsa = mat_to_df(vision_lang_rsa['large_gen_images'], index='vision', columns='lang')

frame_large_lang_rsa = mat_to_df(vision_lang_rsa['large_mid_frames'], index='vision', columns='lang')

# save dataframes
gen_base_lang_rsa.to_csv('results/gen_base_qwen_rsa.csv', index=True)
frame_base_lang_rsa.to_csv('results/frame_base_qwen_rsa.csv', index=True)
gen_large_lang_rsa.to_csv('results/gen_large_qwen_rsa.csv', index=True)
frame_large_lang_rsa.to_csv('results/frame_large_qwen_rsa.csv', index=True)

In [ ]:
# heatmaps
rsa_fig(gen_base_lang_rsa, title='Generated Images RSA', xtitle='Qwen3-1.7B', ytitle='DINOv2 Base')

rsa_fig(frame_base_lang_rsa, title='Middle Frames RSA', xtitle='Qwen3-1.7B', ytitle='DINOv2 Base')

rsa_fig(gen_large_lang_rsa, title='Generated Images RSA', xtitle='Qwen3-1.7B', ytitle='DINOv2 Large')

rsa_fig(frame_large_lang_rsa, title='Middle Frames RSA', xtitle='Qwen3-1.7B', ytitle='DINOv2 Large')

In [ ]:
# heatmaps for paper

# vision language alignment
# gen images base | frame base
# gen images large | frame large

vision_lang_dfs = [gen_base_lang_rsa, frame_base_lang_rsa, gen_large_lang_rsa, frame_large_lang_rsa]

vmin = min(df.min().min() for df in vision_lang_dfs)
vmax = max(df.max().max() for df in vision_lang_dfs)

fig, axes = plt.subplots(2, 2, figsize=(16,8))

# top left
sns.heatmap(gen_base_lang_rsa, ax=axes[0,0], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[0,0].set_title('Generated Images')
axes[0,0].set_xlabel('Qwen3-1.7B')
axes[0,0].set_ylabel('DINOv2 Base')

xlabels = [tick.get_text() for tick in axes[0,0].get_xticklabels()]
new_xlabels = [label.replace("lang_", "") for label in xlabels]
axes[0,0].set_xticklabels(new_xlabels, rotation=0)

base_ylabels = [tick.get_text() for tick in axes[0,0].get_yticklabels()]
new_base_ylabels = [label.replace('vision_', '') for label in base_ylabels]
axes[0,0].set_yticklabels(new_base_ylabels, rotation=0)

# top right
sns.heatmap(frame_base_lang_rsa, ax=axes[0,1], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[0,1].set_title('Middle Frames')
axes[0,1].set_xlabel('Qwen3-1.7B')
axes[0,1].set_ylabel('DINOv2 Base')
axes[0,1].set_xticklabels(new_xlabels, rotation=0)
axes[0,1].set_yticklabels(new_base_ylabels, rotation=0)

# bottom left
sns.heatmap(gen_large_lang_rsa, ax=axes[1,0], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[1,0].set_title('Generated Images')
axes[1,0].set_xlabel('Qwen3-1.7B')
axes[1,0].set_ylabel('DINOv2 Large')
axes[1,0].set_xticklabels(new_xlabels, rotation=0)

large_ylabels = [tick.get_text() for tick in axes[1,0].get_yticklabels()]
new_large_ylabels = [label.replace('vision_', '') for label in large_ylabels]
axes[1,0].set_yticklabels(new_large_ylabels, rotation=0)

# bottom right
sns.heatmap(frame_large_lang_rsa, ax=axes[1,1], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[1,1].set_title('Middle Frames')
axes[1,1].set_xlabel('Qwen3-1.7B')
axes[1,1].set_ylabel('DINOv2 Large')
axes[1,1].set_xticklabels(new_xlabels, rotation=0)
axes[1,1].set_yticklabels(new_large_ylabels, rotation=0)

plt.tight_layout()
plt.savefig('/Users/alexis/Desktop/clean_capstone/figs_paper/vision_lang_rsa.png')
plt.show()

NameError: name 'gen_base_lang_rsa' is not defined

# Audio and Language Alignment

## Linear Predictivity

In [ ]:
audio3_lang_lp_mat = lin_pred_matrix(qwen3, audio_3)
audio3_plus_lang_lp_mat = lin_pred_matrix(qwen3, audio_3_plus)

# shape of matrices
audio3_lang_lp_mat.shape, audio3_plus_lang_lp_mat.shape

In [ ]:
# dataframes for visualizations
audio3_lang_lp = mat_to_df(audio3_lang_lp_mat, index='audio', columns='lang')

audio3_plus_lang_lp = mat_to_df(audio3_plus_lang_lp_mat, index='audio', columns='lang')

# save dataframes
audio3_lang_lp.to_csv('results/audio3_qwen_lp.csv', index=True)
audio3_plus_lang_lp.to_csv('results/audio3_plus_qwen_lp.csv', index=True)

In [ ]:
# heatmaps
lp_fig(audio3_lang_lp, title="Audio Lang LP", xtitle="Qwen3-1.7B", ytitle="BEATs iter3", filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/audio3_lang_lp.png')
lp_fig(audio3_plus_lang_lp, title="Audio Lang LP", xtitle="Qwen3-1.7B", ytitle="BEATs iter3+", filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/audio3_plus_lang_lp.png')

In [ ]:
# heatmaps for paper

# audio language alignment
# beats iter3 | beats iter3+

audio_lang_dfs = [audio3_lang_lp, audio3_plus_lang_lp]

vmin = min(df.min().min() for df in audio_lang_dfs)
vmax = max(df.max().max() for df in audio_lang_dfs)

fig, axes = plt.subplots(1, 2, figsize=(16,5))

# top left
sns.heatmap(audio3_lang_lp, ax=axes[0], cmap='viridis', cbar=True)
# axes[0].set_title('BEATs Iter3')
axes[0].set_xlabel('Qwen3-1.7B')
axes[0].set_ylabel('BEATs Iter3')

xlabels = [tick.get_text() for tick in axes[0].get_xticklabels()]
new_xlabels = [label.replace("lang_", "") for label in xlabels]
axes[0].set_xticklabels(new_xlabels, rotation=0)

base_ylabels = [tick.get_text() for tick in axes[0].get_yticklabels()]
new_base_ylabels = [label.replace('audio_', '') for label in base_ylabels]
axes[0].set_yticklabels(new_base_ylabels, rotation=0)

# top right
sns.heatmap(audio3_plus_lang_lp, ax=axes[1], cmap='viridis', cbar=True)
# axes[1].set_title('BEATs Iter3 Plus')
axes[1].set_xlabel('Qwen3-1.7B')
axes[1].set_ylabel('BEATs Iter3+')
axes[1].set_xticklabels(new_xlabels, rotation=0)
axes[1].set_yticklabels(new_base_ylabels, rotation=0)

plt.tight_layout()
plt.savefig('/Users/alexis/Desktop/clean_capstone/figs_paper/audio_lang_lp.png')
plt.show()

## RSA

In [ ]:
audio3_lang_rsa_mat = rsa_matrix(audio_3, qwen3)
audio3_plus_lang_rsa_mat = rsa_matrix(audio_3_plus, qwen3)

# shape of matrices
audio3_lang_rsa_mat.shape, audio3_plus_lang_rsa_mat.shape

In [ ]:
# dataframes for visualizations
audio3_lang_rsa = mat_to_df(audio3_lang_rsa_mat, index='audio', columns='lang')

audio3_plus_lang_rsa = mat_to_df(audio3_plus_lang_rsa_mat, index='audio', columns='lang')

# save dataframes
audio3_lang_rsa.to_csv("results/audio3_qwen_rsa.csv", index=True)
audio3_plus_lang_rsa.to_csv("results/audio3_plus_qwen_rsa.csv", index=True)

In [ ]:
# heatmaps
rsa_fig(audio3_lang_rsa, title='Audio Lang RSA', xtitle='Qwen3-1.7B', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/audio3_lang_rsa.png')
rsa_fig(audio3_plus_lang_rsa, title='Audio Lang RSA', xtitle='Qwen3-1.7B', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/audio3_plus_lang_rsa.png')

In [ ]:
# heatmaps for paper

# audio language alignment
# beats iter3 | beats iter3+

audio_lang_dfs = [audio3_lang_rsa, audio3_plus_lang_rsa]

vmin = min(df.min().min() for df in audio_lang_dfs)
vmax = max(df.max().max() for df in audio_lang_dfs)

fig, axes = plt.subplots(1, 2, figsize=(16,5))

# top left
sns.heatmap(audio3_lang_rsa, ax=axes[0], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
# axes[0].set_title('BEATs Iter3')
axes[0].set_xlabel('Qwen3-1.7B')
axes[0].set_ylabel('BEATs Iter3')

xlabels = [tick.get_text() for tick in axes[0].get_xticklabels()]
new_xlabels = [label.replace("lang_", "") for label in xlabels]
axes[0].set_xticklabels(new_xlabels, rotation=0)

base_ylabels = [tick.get_text() for tick in axes[0].get_yticklabels()]
new_base_ylabels = [label.replace('audio_', '') for label in base_ylabels]
axes[0].set_yticklabels(new_base_ylabels, rotation=0)

# top right
sns.heatmap(audio3_plus_lang_rsa, ax=axes[1], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
# axes[1].set_title('BEATs Iter3 Plus')
axes[1].set_xlabel('Qwen3-1.7B')
axes[1].set_ylabel('BEATs Iter3+')
axes[1].set_xticklabels(new_xlabels, rotation=0)
axes[1].set_yticklabels(new_base_ylabels, rotation=0)

plt.tight_layout()
plt.savefig('/Users/alexis/Desktop/clean_capstone/figs_paper/audio_lang_rsa.png')
plt.show()

# Vision and Audio Alignment

## Linear Predictivity

In [ ]:
vision_audio_lp = {
    'gen_base_audio3': lin_pred_matrix(gen_base, audio_3),
    'gen_base_audio3_plus': lin_pred_matrix(gen_base, audio_3_plus),
    'frame_base_audio3': lin_pred_matrix(frame_base, audio_3),
    'frame_base_audio3_plus': lin_pred_matrix(frame_base, audio_3_plus),
    'gen_large_audio3': lin_pred_matrix(gen_large, audio_3),
    'gen_large_audio3_plus': lin_pred_matrix(gen_large, audio_3_plus),
    'frame_large_audio3': lin_pred_matrix(frame_large, audio_3),
    'frame_large_audio3_plus': lin_pred_matrix(frame_large, audio_3_plus)
}

In [ ]:
# dataframes for visualization
gen_base_audio3_lp = mat_to_df(vision_audio_lp['gen_base_audio3'], index='audio', columns='vision')
gen_base_audio3_plus_lp = mat_to_df(vision_audio_lp['gen_base_audio3_plus'], index='audio', columns='vision')
frame_base_audio3_lp = mat_to_df(vision_audio_lp['frame_base_audio3'], index='audio', columns='vision')
frame_base_audio3_plus_lp = mat_to_df(vision_audio_lp['frame_base_audio3_plus'], index='audio', columns='vision')
gen_large_audio3_lp = mat_to_df(vision_audio_lp['gen_large_audio3'], index='audio', columns='vision')
gen_large_audio3_plus_lp = mat_to_df(vision_audio_lp['gen_large_audio3_plus'], index='audio', columns='vision')
frame_large_audio3_lp = mat_to_df(vision_audio_lp['frame_large_audio3'], index='audio', columns='vision')
frame_large_audio3_plus_lp = mat_to_df(vision_audio_lp['frame_large_audio3_plus'], index='audio', columns='vision')

# save dataframes
gen_base_audio3_lp.to_csv('results/gen_base_audio3_lp.csv', index=True)
gen_base_audio3_plus_lp.to_csv('results/gen_base_audio3_plus_lp.csv', index=True)
frame_base_audio3_lp.to_csv('results/frame_base_audio3_lp.csv', index=True)
frame_base_audio3_plus_lp.to_csv('results/frame_base_audio3_plus_lp.csv', index=True)
gen_large_audio3_lp.to_csv('results/gen_large_audio3_lp.csv', index=True)
gen_large_audio3_plus_lp.to_csv('results/gen_large_audio3_plus_lp.csv', index=True)
frame_large_audio3_lp.to_csv('results/frame_large_audio3_lp.csv', index=True)
frame_large_audio3_plus_lp.to_csv('results/frame_large_audio3_plus_lp.csv', index=True)


In [ ]:
# heatmaps
lp_fig(gen_base_audio3_lp, title='Generated Images LP', xtitle='DINOv2 Base', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_base_audio3_lp.png')
lp_fig(gen_base_audio3_plus_lp, title='Generated Images LP', xtitle='DINOv2 Base', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_base_audio3_plus_lp.png')

lp_fig(frame_base_audio3_lp, title='Middle Frames LP', xtitle='DINOv2 Base', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_base_audio3_lp.png')
lp_fig(frame_base_audio3_plus_lp, title='Middle Frames LP', xtitle='DINOv2 Base', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_base_audio3_plus_lp.png')

lp_fig(gen_large_audio3_lp, title='Generated Images LP', xtitle='DINOv2 Large', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_large_audio3_lp.png')
lp_fig(gen_large_audio3_plus_lp, title='Generated Images LP', xtitle='DINOv2 Large', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_large_audio3_plus_lp.png')

lp_fig(frame_large_audio3_lp, title='Middle Frames LP', xtitle='DINOv2 Large', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_large_audio3_lp.png')
lp_fig(frame_large_audio3_plus_lp, title='Middle Frames LP', xtitle='DINOv2 Large', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_large_audio3_plus_lp.png')

NameError: name 'gen_base_audio3_lp' is not defined

In [ ]:
# heatmaps for paper

# vision audio alignment
# base row: gen_iter3, frame_iter3, gen_iter3+, frame_iter3+
# large row: gen_iter3, frame_iter3, gen_iter3+, frame_iter3+

vision_audio_dfs = [gen_base_audio3_lp, gen_base_audio3_plus_lp, frame_base_audio3_lp, frame_base_audio3_plus_lp, gen_large_audio3_lp, gen_large_audio3_plus_lp, frame_large_audio3_lp, frame_large_audio3_plus_lp]

vmin = min(df.min().min() for df in vision_audio_dfs)
vmax = max(df.max().max() for df in vision_audio_dfs)

fig, axes = plt.subplots(2, 4, figsize=(23,9))

### row 1: dinov2 base
sns.heatmap(gen_base_audio3_lp, ax=axes[0, 0], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[0,0].set_title('Generated Images')
axes[0,0].set_xlabel('DINOv2 Base')
axes[0,0].set_ylabel('BEATs Iter3')
axes[0,0].set_xticklabels(range(gen_base_audio3_lp.shape[1]), rotation=0)
axes[0,0].set_yticklabels(range(gen_base_audio3_lp.shape[0])[::-1], rotation=0)

sns.heatmap(frame_base_audio3_lp, ax=axes[0, 1], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[0,1].set_title('Middle Frames')
axes[0,1].set_xlabel('DINOv2 Base')
axes[0,1].set_ylabel('BEATs Iter3')
axes[0,1].set_xticklabels(range(frame_base_audio3_lp.shape[1]), rotation=0)
axes[0,1].set_yticklabels(range(frame_base_audio3_lp.shape[0])[::-1], rotation=0)

sns.heatmap(gen_base_audio3_plus_lp, ax=axes[0, 2], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[0,2].set_title('Generated Images')
axes[0,2].set_xlabel('DINOv2 Base')
axes[0,2].set_ylabel('BEATs Iter3+')
axes[0,2].set_xticklabels(range(gen_base_audio3_plus_lp.shape[1]), rotation=0)
axes[0,2].set_yticklabels(range(gen_base_audio3_plus_lp.shape[0])[::-1], rotation=0)

sns.heatmap(frame_base_audio3_plus_lp, ax=axes[0, 3], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[0,3].set_title('Middle Frames')
axes[0,3].set_xlabel('DINOv2 Base')
axes[0,3].set_ylabel('BEATs Iter3+')
axes[0,3].set_xticklabels(range(frame_base_audio3_plus_lp.shape[1]), rotation=0)
axes[0,3].set_yticklabels(range(frame_base_audio3_plus_lp.shape[0])[::-1], rotation=0)


### row 2: dinov2 large
sns.heatmap(gen_large_audio3_lp, ax=axes[1, 0], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[1,0].set_title('Generated Images')
axes[1,0].set_xlabel('DINOv2 Large')
axes[1,0].set_ylabel('BEATs Iter3')
axes[1,0].set_yticklabels(range(gen_large_audio3_lp.shape[0])[::-1], rotation=0)

large_xlabels = [tick.get_text() for tick in axes[1,0].get_xticklabels()]
new_xlabels = [label.replace("vision_", "") for label in large_xlabels]
axes[1,0].set_xticklabels(new_xlabels, rotation=0)

sns.heatmap(frame_large_audio3_lp, ax=axes[1, 1], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[1,1].set_title('Middle Frames')
axes[1,1].set_xlabel('DINOv2 Large')
axes[1,1].set_ylabel('BEATs Iter3')
axes[1,1].set_xticklabels(new_xlabels, rotation=0)
axes[1,1].set_yticklabels(range(frame_large_audio3_lp.shape[0])[::-1], rotation=0)

sns.heatmap(gen_large_audio3_plus_lp, ax=axes[1, 2], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[1,2].set_title('Generated Images')
axes[1,2].set_xlabel('DINOv2 Large')
axes[1,2].set_ylabel('BEATs Iter3+')
axes[1,2].set_xticklabels(new_xlabels, rotation=0)
axes[1,2].set_yticklabels(range(gen_large_audio3_plus_lp.shape[0])[::-1], rotation=0)

sns.heatmap(frame_large_audio3_plus_lp, ax=axes[1, 3], vmin=vmin, vmax=vmax, cmap='viridis', cbar=True)
axes[1,3].set_title('Middle Frames')
axes[1,3].set_xlabel('DINOv2 Large')
axes[1,3].set_ylabel('BEATs Iter3+')
axes[1,3].set_xticklabels(new_xlabels, rotation=0)
axes[1,3].set_yticklabels(range(gen_large_audio3_plus_lp.shape[0])[::-1], rotation=0)

plt.tight_layout()
plt.savefig('/Users/alexis/Desktop/clean_capstone/figs_paper/vision_audio_lp.png')
plt.show()

NameError: name 'gen_base_audio3_lp' is not defined

## RSA

In [ ]:
vision_audio_rsa = {
    'gen_base_audio3': rsa_matrix(audio_3, gen_base),
    'gen_base_audio3_plus': rsa_matrix(audio_3_plus, gen_base),
    'frame_base_audio3': rsa_matrix(audio_3, frame_base),
    'frame_base_audio3_plus': rsa_matrix(audio_3_plus, frame_base),
    'gen_large_audio3': rsa_matrix(audio_3, gen_large),
    'gen_large_audio3_plus': rsa_matrix(audio_3_plus, gen_large),
    'frame_large_audio3': rsa_matrix(audio_3, frame_large),
    'frame_large_audio3_plus': rsa_matrix(audio_3_plus, frame_large)
}

In [ ]:
# dataframes for visualization
gen_base_audio3_rsa = mat_to_df(vision_audio_rsa['gen_base_audio3'], index='audio', columns='vision')
gen_base_audio3_plus_rsa = mat_to_df(vision_audio_rsa['gen_base_audio3_plus'], index='audio', columns='vision')
frame_base_audio3_rsa = mat_to_df(vision_audio_rsa['frame_base_audio3'], index='audio', columns='vision')
frame_base_audio3_plus_rsa = mat_to_df(vision_audio_rsa['frame_base_audio3_plus'], index='audio', columns='vision')
gen_large_audio3_rsa = mat_to_df(vision_audio_rsa['gen_large_audio3'], index='audio', columns='vision')
gen_large_audio3_plus_rsa = mat_to_df(vision_audio_rsa['gen_large_audio3_plus'], index='audio', columns='vision')
frame_large_audio3_rsa = mat_to_df(vision_audio_rsa['frame_large_audio3'], index='audio', columns='vision')
frame_large_audio3_plus_rsa = mat_to_df(vision_audio_rsa['frame_large_audio3_plus'], index='audio', columns='vision')

# save dataframes
gen_base_audio3_rsa.to_csv('results/gen_base_audio3_rsa.csv', index=True)
gen_base_audio3_plus_rsa.to_csv('results/gen_base_audio3_plus_rsa.csv', index=True)
frame_base_audio3_rsa.to_csv('results/frame_base_audio3_rsa.csv', index=True)
frame_base_audio3_plus_rsa.to_csv('results/frame_base_audio3_plus_rsa.csv', index=True)
gen_large_audio3_rsa.to_csv('results/gen_large_audio3_rsa.csv', index=True)
gen_large_audio3_plus_rsa.to_csv('results/gen_large_audio3_plus_rsa.csv', index=True)
frame_large_audio3_rsa.to_csv('results/frame_large_audio3_rsa.csv', index=True)
frame_large_audio3_plus_rsa.to_csv('results/frame_large_audio3_plus_rsa.csv', index=True)

NameError: name 'vision_audio_rsa' is not defined

In [ ]:
# heatmaps
rsa_fig(gen_base_audio3_rsa, title='Generated Images RSA', xtitle='DINOv2 Base', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_base_audio3_rsa.png')
rsa_fig(gen_base_audio3_plus_rsa, title='Generated Images RSA', xtitle='DINOv2 Base', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_base_audio3_plus_rsa.png')

rsa_fig(frame_base_audio3_rsa, title='Middle Frames RSA', xtitle='DINOv2 Base', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_base_audio3_rsa.png')
rsa_fig(frame_base_audio3_plus_rsa, title='Middle Frames RSA', xtitle='DINOv2 Base', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_base_audio3_plus_rsa.png')

rsa_fig(gen_large_audio3_rsa, title='Generated Images RSA', xtitle='DINOv2 Large', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_large_audio3_rsa.png')
rsa_fig(gen_large_audio3_plus_rsa, title='Generated Images RSA', xtitle='DINOv2 Large', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/gen_large_audio3_plus_rsa.png')

rsa_fig(frame_large_audio3_rsa, title='Middle Frames RSA', xtitle='DINOv2 Large', ytitle='BEATs iter3', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_large_audio3_rsa.png')
rsa_fig(frame_large_audio3_plus_rsa, title='Middle Frames RSA', xtitle='DINOv2 Large', ytitle='BEATs iter3+', filename='/Users/alexis/Desktop/clean_capstone/figs_annotated/frame_large_audio3_plus_rsa.png')

In [ ]:
# heatmaps for paper

# vision audio alignment
# base row: gen_iter3, frame_iter3, gen_iter3+, frame_iter3+
# large row: gen_iter3, frame_iter3, gen_iter3+, frame_iter3+

vision_audio_dfs = [gen_base_audio3_rsa, gen_base_audio3_plus_rsa, frame_base_audio3_rsa, frame_base_audio3_plus_rsa, gen_large_audio3_rsa, gen_large_audio3_plus_rsa, frame_large_audio3_rsa, frame_large_audio3_plus_rsa]

vmin = min(df.min().min() for df in vision_audio_dfs)
vmax = max(df.max().max() for df in vision_audio_dfs)

fig, axes = plt.subplots(2, 4, figsize=(23,9))

### row 1: dinov2 base
sns.heatmap(gen_base_audio3_rsa, ax=axes[0, 0], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[0,0].set_title('Generated Images')
axes[0,0].set_xlabel('DINOv2 Base')
axes[0,0].set_ylabel('BEATs Iter3')
axes[0,0].set_xticklabels(range(gen_base_audio3_rsa.shape[1]), rotation=0)
axes[0,0].set_yticklabels(range(gen_base_audio3_rsa.shape[0])[::-1], rotation=0)

sns.heatmap(frame_base_audio3_rsa, ax=axes[0, 1], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[0,1].set_title('Middle Frames')
axes[0,1].set_xlabel('DINOv2 Base')
axes[0,1].set_ylabel('BEATs Iter3')
axes[0,1].set_xticklabels(range(frame_base_audio3_rsa.shape[1]), rotation=0)
axes[0,1].set_yticklabels(range(frame_base_audio3_rsa.shape[0])[::-1], rotation=0)

sns.heatmap(gen_base_audio3_plus_rsa, ax=axes[0, 2], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[0,2].set_title('Generated Images')
axes[0,2].set_xlabel('DINOv2 Base')
axes[0,2].set_ylabel('BEATs Iter3+')
axes[0,2].set_xticklabels(range(gen_base_audio3_plus_rsa.shape[1]), rotation=0)
axes[0,2].set_yticklabels(range(gen_base_audio3_plus_rsa.shape[0])[::-1], rotation=0)

sns.heatmap(frame_base_audio3_plus_rsa, ax=axes[0, 3], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[0,3].set_title('Middle Frames')
axes[0,3].set_xlabel('DINOv2 Base')
axes[0,3].set_ylabel('BEATs Iter3+')
axes[0,3].set_xticklabels(range(frame_base_audio3_plus_rsa.shape[1]), rotation=0)
axes[0,3].set_yticklabels(range(frame_base_audio3_plus_rsa.shape[0])[::-1], rotation=0)


### row 2: dinov2 large
sns.heatmap(gen_large_audio3_rsa, ax=axes[1, 0], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[1,0].set_title('Generated Images')
axes[1,0].set_xlabel('DINOv2 Large')
axes[1,0].set_ylabel('BEATs Iter3')
axes[1,0].set_yticklabels(range(gen_large_audio3_rsa.shape[0])[::-1], rotation=0)

large_xlabels = [tick.get_text() for tick in axes[1,0].get_xticklabels()]
new_xlabels = [label.replace("vision_", "") for label in large_xlabels]
axes[1,0].set_xticklabels(new_xlabels, rotation=0)

sns.heatmap(frame_large_audio3_rsa, ax=axes[1, 1], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[1,1].set_title('Middle Frames')
axes[1,1].set_xlabel('DINOv2 Large')
axes[1,1].set_ylabel('BEATs Iter3')
axes[1,1].set_xticklabels(new_xlabels, rotation=0)
axes[1,1].set_yticklabels(range(frame_large_audio3_rsa.shape[0])[::-1], rotation=0)

sns.heatmap(gen_large_audio3_plus_rsa, ax=axes[1, 2], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[1,2].set_title('Generated Images')
axes[1,2].set_xlabel('DINOv2 Large')
axes[1,2].set_ylabel('BEATs Iter3+')
axes[1,2].set_xticklabels(new_xlabels, rotation=0)
axes[1,2].set_yticklabels(range(gen_large_audio3_plus_rsa.shape[0])[::-1], rotation=0)

sns.heatmap(frame_large_audio3_plus_rsa, ax=axes[1, 3], vmin=vmin, vmax=vmax, cmap='magma', cbar=True)
axes[1,3].set_title('Middle Frames')
axes[1,3].set_xlabel('DINOv2 Large')
axes[1,3].set_ylabel('BEATs Iter3+')
axes[1,3].set_xticklabels(new_xlabels, rotation=0)
axes[1,3].set_yticklabels(range(frame_large_audio3_plus_rsa.shape[0])[::-1], rotation=0)

plt.tight_layout()
plt.savefig('/Users/alexis/Desktop/clean_capstone/figs_paper/vision_audio_rsa.png')
plt.show()

NameError: name 'gen_base_audio3_rsa' is not defined

In [ ]:
# zip result csv files
shutil.make_archive('alignment_results', 'zip', '/content/results')

In [ ]:
# using lin_pred_matrix_folds()
lp_fig(df=frame_large_audio3_plus_lp, title="Middle Frame LP", xtitle="DINOv2 Large", ytitle="BEATs iter3+")